# Consuming Another Domain's Data Product on AWS


## Instructions


In this exercise, you willpractice consuming data from another domain by reading a data contract, identifying available tables and keys, connecting to the data, running queries, answering a cross-domain business question, and documenting results.

---

## Scenario

Your team owns the **Sales Orders** data product.

Another team owns the **Inventory & Distribution** data product.

You have been asked to answer the following business question:

> Which distribution center generated the most revenue in 2025?

To answer this, you need data from both domains.

---




## Provided Data Products

### Sales Orders Domain

The Sales Orders domain includes:

- `orders`
- `order_items`

Important fields:

- `orders.order_id`
- `orders.order_date`
- `orders.order_status`
- `order_items.order_id`
- `order_items.inventory_id`
- `order_items.line_total`

---

### Inventory & Distribution Domain

The Inventory & Distribution domain includes:

- `inventory`
- `distribution_center`

Important fields:

- `inventory.inventory_id`
- `inventory.distribution_center_id`
- `distribution_center.distribution_center_id`
- `distribution_center.distribution_center_name`
- `distribution_center.region`

---

## Step 1: Read the Other Domain's Contract

Review the Inventory & Distribution data contract.

Identify:

- What tables are available?
- What fields are available?
- What primary keys exist?
- What foreign keys exist?
- Which fields can connect to the Sales Orders domain?



---


### Data Contract A: Sales Orders Domain

In [ ]:
{
  "contract_name": "Sales Orders Data Product Contract",
  "version": "1.0",
  "description": "Defines customer orders and order line items for retail sales transactions.",
  "tables": [
    {
      "table_name": "orders",
      "description": "Order header records.",
      "columns": [
        {"name": "order_id", "data_type": "string", "required": true, "primary_key": true},
        {"name": "customer_id", "data_type": "string", "required": true},
        {"name": "order_date", "data_type": "date", "required": true},
        {"name": "order_channel", "data_type": "string", "required": true},
        {"name": "order_status", "data_type": "string", "required": true}
      ]
    },
    {
      "table_name": "order_items",
      "description": "Line-level products included in each order.",
      "columns": [
        {"name": "order_item_id", "data_type": "string", "required": true, "primary_key": true},
        {"name": "order_id", "data_type": "string", "required": true, "foreign_key": {"table": "orders", "column": "order_id"}},
        {"name": "inventory_id", "data_type": "string", "required": true},
        {"name": "quantity_ordered", "data_type": "integer", "required": true},
        {"name": "unit_price_at_purchase", "data_type": "decimal", "required": true},
        {"name": "line_total", "data_type": "decimal", "required": true}
      ]
    }
  ],
  "business_rules": [
    "Every order item must belong to a valid order.",
    "Quantity ordered must be greater than zero.",
    "Line total must equal quantity ordered multiplied by unit price at purchase.",
    "Cancelled and returned orders should be excluded from revenue reporting."
  ]
}

### Data Contract B: Inventory & Distribution Domain

In [ ]:
{
  "contract_name": "Inventory Distribution Data Product Contract",
  "version": "1.0",
  "description": "Defines inventory items and the distribution centers responsible for fulfillment.",
  "tables": [
    {
      "table_name": "inventory",
      "description": "Products available for fulfillment.",
      "columns": [
        {"name": "inventory_id", "data_type": "string", "required": true, "primary_key": true},
        {"name": "product_name", "data_type": "string", "required": true},
        {"name": "product_category", "data_type": "string", "required": true},
        {"name": "unit_price", "data_type": "decimal", "required": true},
        {"name": "quantity_on_hand", "data_type": "integer", "required": true},
        {"name": "distribution_center_id", "data_type": "string", "required": true, "foreign_key": {"table": "distribution_center", "column": "distribution_center_id"}}
      ]
    },
    {
      "table_name": "distribution_center",
      "description": "Distribution centers that fulfill inventory.",
      "columns": [
        {"name": "distribution_center_id", "data_type": "string", "required": true, "primary_key": true},
        {"name": "distribution_center_name", "data_type": "string", "required": true},
        {"name": "city", "data_type": "string", "required": true},
        {"name": "state", "data_type": "string", "required": true},
        {"name": "region", "data_type": "string", "required": true}
      ]
    }
  ],
  "business_rules": [
    "Every inventory item must be assigned to a valid distribution center.",
    "Quantity on hand cannot be negative.",
    "Unit price must be greater than zero."
  ]
}

### ERD: Sales Orders Domain

```mermaid
erDiagram

    orders {
        varchar order_id PK
        varchar customer_id
        date order_date
        varchar order_channel
        varchar order_status
    }

    order_items {
        varchar order_item_id PK
        varchar order_id FK
        varchar inventory_id
        integer quantity_ordered
        numeric unit_price_at_purchase
        numeric line_total
    }

    orders ||--o{ order_items : contains

### ERD: Inventory & Distribution Domain

```mermaid

erDiagram

    distribution_center {
        varchar distribution_center_id PK
        varchar distribution_center_name
        varchar city
        char state
        varchar region
    }

    inventory {
        varchar inventory_id PK
        varchar product_name
        varchar product_category
        numeric unit_price
        integer quantity_on_hand
        varchar distribution_center_id FK
    }

    distribution_center ||--o{ inventory : stores


### Step 2
Using the provided starter code:

Connect to AWS

Build Databases

Populate Database

---

## Connect to AWS

In [ ]:
import boto3
import json
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values


AWS_ACCESS_KEY_ID = "<get key from classroom>"
AWS_SECRET_ACCESS_KEY = "<get key from classroom>"
AWS_SESSION_TOKEN = "<get key from classroom>"
AWS_REGION = "us-east-1"

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION,
)


# Get DB connection info from CloudFormation
cf = session.client("cloudformation")
stack = cf.describe_stacks()["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}

db_host = outputs["DBEndpoint"]
db_port = int(outputs["DBPort"])
db_name = outputs["DBName"]
secret_arn = outputs["MasterSecretArn"]

# Get credentials from Secrets Manager
sm = session.client("secretsmanager", region_name="us-east-1")
secret_response = sm.get_secret_value(SecretId=secret_arn)
db_secret = json.loads(secret_response["SecretString"])

# Connect to DB
try:
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_secret["username"],
        password=db_secret["password"],
        sslmode="require",
    )
    print("Successfully connected to the database")
 
except Exception as e:
    print(f"Connection failed: {type(e).__name__}: {e}")

## Create Tables

In [ ]:
conn.rollback()
cur = conn.cursor()

create_tables_sql = """
DROP TABLE IF EXISTS order_items CASCADE;
DROP TABLE IF EXISTS inventory CASCADE;
DROP TABLE IF EXISTS orders CASCADE;
DROP TABLE IF EXISTS distribution_center CASCADE;
CREATE TABLE orders (
    order_id VARCHAR(20) PRIMARY KEY,
    customer_id VARCHAR(20) NOT NULL,
    order_date DATE NOT NULL,
    order_channel VARCHAR(50) NOT NULL,
    order_status VARCHAR(50) NOT NULL
);

CREATE TABLE order_items (
    order_item_id VARCHAR(20) PRIMARY KEY,
    order_id VARCHAR(20) NOT NULL,
    inventory_id VARCHAR(20) NOT NULL,
    quantity_ordered INTEGER NOT NULL,
    unit_price_at_purchase NUMERIC(10,2) NOT NULL,
    line_total NUMERIC(12,2) NOT NULL,
    CONSTRAINT fk_order_items_orders
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id),
    CONSTRAINT chk_quantity_ordered
        CHECK (quantity_ordered > 0),
    CONSTRAINT chk_line_total
        CHECK (line_total > 0)
);

CREATE TABLE distribution_center (
    distribution_center_id VARCHAR(20) PRIMARY KEY,
    distribution_center_name VARCHAR(100) NOT NULL,
    city VARCHAR(75) NOT NULL,
    state CHAR(2) NOT NULL,
    region VARCHAR(50) NOT NULL
);

CREATE TABLE inventory (
    inventory_id VARCHAR(20) PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL,
    product_category VARCHAR(75) NOT NULL,
    unit_price NUMERIC(10,2) NOT NULL,
    quantity_on_hand INTEGER NOT NULL,
    distribution_center_id VARCHAR(20) NOT NULL,
    CONSTRAINT fk_inventory_distribution_center
        FOREIGN KEY (distribution_center_id)
        REFERENCES distribution_center(distribution_center_id),
    CONSTRAINT chk_unit_price
        CHECK (unit_price > 0),
    CONSTRAINT chk_quantity_on_hand
        CHECK (quantity_on_hand >= 0)
);
"""

cur.execute(create_tables_sql)
conn.commit()
cur.close()

print("Cross-domain tables created successfully.")

## Load CSV files into memory

In [ ]:
import pandas as pd
from psycopg2.extras import execute_values

conn.rollback()

orders_df = pd.read_csv("../cross-domain-data-product-exercise/udacity_orders.csv")
order_items_df = pd.read_csv("../cross-domain-data-product-exercise/udacity_order_items.csv")
inventory_df = pd.read_csv("../cross-domain-data-product-exercise/udacity_inventory.csv")
distribution_center_df = pd.read_csv("../cross-domain-data-product-exercise/udacity_distribution_center.csv")

print("orders:", len(orders_df))
print("order_items:", len(order_items_df))
print("inventory:", len(inventory_df))
print("distribution_center:", len(distribution_center_df))

## Insert Data into tables

In [ ]:
conn.rollback()
cur = conn.cursor()

tables = [
    (
        "orders",
        ["order_id", "customer_id", "order_date", "order_channel", "order_status"],
        orders_df,
        lambda row: (row.order_id, row.customer_id, row.order_date, row.order_channel, row.order_status)
    ),
    (
        "distribution_center",
        ["distribution_center_id", "distribution_center_name", "city", "state", "region"],
        distribution_center_df,
        lambda row: (row.distribution_center_id, row.distribution_center_name, row.city, row.state, row.region)
    ),
    (
        "inventory",
        ["inventory_id", "product_name", "product_category", "unit_price", "quantity_on_hand", "distribution_center_id"],
        inventory_df,
        lambda row: (row.inventory_id, row.product_name, row.product_category, float(row.unit_price), int(row.quantity_on_hand), row.distribution_center_id)
    ),
    (
        "order_items",
        ["order_item_id", "order_id", "inventory_id", "quantity_ordered", "unit_price_at_purchase", "line_total"],
        order_items_df,
        lambda row: (row.order_item_id, row.order_id, row.inventory_id, int(row.quantity_ordered), float(row.unit_price_at_purchase), float(row.line_total))
    ),
]

for table, cols, df, row_fn in tables:
    sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES %s;"
    values = [row_fn(row) for row in df.itertuples(index=False)]
    execute_values(cur, sql, values)
    print(f"{table} loaded.")

conn.commit()
cur.close()

print ("data loaded")

## Validate Data Load

In [ ]:
cur = conn.cursor()

cur.execute("""
SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'inventory', COUNT(*) FROM inventory
UNION ALL
SELECT 'distribution_center', COUNT(*) FROM distribution_center;
""")

for row in cur.fetchall():
    print(row)

cur.close()

## Question 1: Which field connects the Sales Orders domain to the Inventory & Distribution domain?

Answer in text box below

## Question 2: Which distribution center generated the most revenue in 2025?

Use Python code for this answer

## Question 3: What is total revenue by distribution center?

Use Python code for this answer

## Question 4: Which product category generated the most revenue?

Use Python code for this answer

## Question 5: Create a serving-layer view for consumers.

Use Python code for this answer

## Create a Cross-Domain ERD

Use Mermaid code to create ERD below

```mermaid